# YOLO11n cumulative DySample / PLS / SCAM experiments

This notebook runs exactly one independently initialized experiment at a time:

1. `incdw_dysample`
2. `incdw_dysample_pls`
3. `incdw_dysample_pls_scam`

Every run inherits compatible tensors from the same official `yolo11n.pt`.
It never loads another experiment's `best.pt`. The preflight runs first, then
`RUN_TRAINING=True` starts the official Ultralytics `YOLO.train(...)` path.

## 1. Install the pinned environment

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics==8.4.92", "tqdm", "pyyaml"],
    check=True,
)
import ultralytics
assert ultralytics.__version__ == "8.4.92", ultralytics.__version__
print("Ultralytics:", ultralytics.__version__)

## 2. Mount Drive and clone the private experiment branch

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import base64
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoverdZ/ship-yolo.git"
BRANCH = "feature/dysample-pls-scam-cumulative"
REPO_ROOT = Path("/content/ship-yolo")
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError("Create a private Colab secret named GITHUB_TOKEN.")

basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git = ["git", "-c", f"http.extraHeader=AUTHORIZATION: basic {basic}"]
if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(
        [*git, "clone", "--branch", BRANCH, "--single-branch",
         REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run([*git, "-C", str(REPO_ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "switch", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_ROOT)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()
print("Repository:", REPO_ROOT)
print("Branch:", BRANCH)
print("Commit:", commit)

## 3. Copy the Drive dataset to local Colab storage

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))

from colab.cumulative_training import (
    COPY_WORKERS,
    copy_dataset_to_local,
    create_local_data_yaml,
)

copy_report = copy_dataset_to_local(workers=COPY_WORKERS)
DATA_YAML = create_local_data_yaml("data.yaml")

## 4. Choose one model and run the CPU preflight

In [ ]:
EXPERIMENT = "incdw_dysample"
# Alternatives:
# EXPERIMENT = "incdw_dysample_pls"
# EXPERIMENT = "incdw_dysample_pls_scam"

RUN_NAMES = {
    "incdw_dysample": "yolo11n_incdw_dysample_640",
    "incdw_dysample_pls": "yolo11n_incdw_dysample_pls_640",
    "incdw_dysample_pls_scam": "yolo11n_incdw_dysample_pls_scam_640",
}
if EXPERIMENT not in RUN_NAMES:
    raise ValueError(f"Unknown experiment: {EXPERIMENT}")
RUN_NAME = RUN_NAMES[EXPERIMENT]
print("Selected:", EXPERIMENT, "->", RUN_NAME)

In [ ]:
from pathlib import Path

AUDIT_PATH = Path(
    f"/content/drive/MyDrive/ship_detection/audits/{RUN_NAME}_preflight.json"
)
subprocess.run(
    [
        sys.executable,
        "tools/check_cumulative_models.py",
        "--model", EXPERIMENT,
        "--weights", "yolo11n.pt",
        "--imgsz", "640",
        "--output", str(AUDIT_PATH),
    ],
    check=True,
)
print("Preflight report:", AUDIT_PATH)

## 5. Train only after reviewing the preflight

In [ ]:
from tools.train_cumulative_models import train_experiment

RUN_TRAINING = True
if not RUN_TRAINING:
    print("Training is disabled.")
else:
    # Direct call in the current Colab kernel: per-epoch logs and progress
    # bars remain visible in this cell. Never launch training in a subprocess.
    train_experiment(
        experiment=EXPERIMENT,
        data=DATA_YAML,
        weights="yolo11n.pt",
        project="/content/drive/MyDrive/ship_detection/runs",
        name=RUN_NAME,
        device="0",
        epochs=150,
        imgsz=640,
        batch=8,
        workers=2,
        cache="disk",
        # CUDA grid_sample backward used by DySample is non-deterministic.
        # False prevents PyTorch warning stacks from breaking the tqdm display.
        deterministic=False,
    )

## Checks and next steps

- Confirm the inheritance report says `passed: true` and record
  `inherited_tensors / target_state_tensors`.
- Each run name must be unused. The training entrypoint rejects a directory
  containing prior training artifacts.
- Run the next experiment by changing only `EXPERIMENT`; every model still
  starts from official `yolo11n.pt`.
- Keep the test set sealed. Ultralytics trains against the unchanged training
  split and validates on the unchanged validation split from `DATA_YAML`.